# DS2002 · Pandas Core Ops

**Studio — 2026-09-16 · Fall 2026**  
**Class time:** 45 minutes

---

## Building a vendor report

Today you build one deliverable end to end: a per-vendor summary that a game-day manager could act on. Three sources, a join that does not behave, and a report at the end.

The data has problems planted in it. Finding them is part of the work — a report you cannot defend is worth nothing, however good the code looks.

In [1]:
import pandas as pd
from io import StringIO

orders = pd.read_csv(StringIO('''order_id,vendor_id,item,qty,price
1,V-01,Cheeseburger,2,7.50
2,V-10,Foam Finger,1,12.00
3,V-01,Hot Dog,3,4.50
4,V-18,Rain Poncho,5,6.00
5,V-10,UVA T-Shirt,1,24.00
6,V-05,Chicken Tacos,4,6.50
7,V-18,Rain Poncho,8,6.00
8,V-42,Kettle Corn,3,5.00'''))

vendors = pd.read_csv(StringIO('''vendor_id,vendor_name,zone
V-01,Hoos Burgers,A
V-05,Rotunda Tacos,B
V-10,Cav Merch North,A
V-18,Rally Rain Gear,C
V-18,Rally Rain Gear,C'''))

targets = pd.read_csv(StringIO('''zone,revenue_target
A,80
B,25
C,60'''))

print('orders:', orders.shape, '| vendors:', vendors.shape, '| targets:', targets.shape)
orders

orders: (8, 5) | vendors: (5, 3) | targets: (3, 2)


,order_id,vendor_id,item,qty,price
0,1,V-01,Cheeseburger,2,7.5
1,2,V-10,Foam Finger,1,12.0
2,3,V-01,Hot Dog,3,4.5
3,4,V-18,Rain Poncho,5,6.0
4,5,V-10,UVA T-Shirt,1,24.0
5,6,V-05,Chicken Tacos,4,6.5
6,7,V-18,Rain Poncho,8,6.0
7,8,V-42,Kettle Corn,3,5.0


### Worked example — the merge, done carefully

Here is one merge done properly, so the pattern is on the screen before you write anything. Three things happen: record the baseline, merge with `indicator=True`, then compare against the baseline.

In [2]:
baseline_rows = len(orders)
orders['revenue'] = orders['qty'] * orders['price']
baseline_revenue = orders['revenue'].sum()
print(f'before: {baseline_rows} rows, ${baseline_revenue:.2f}')

check = orders.merge(vendors, on='vendor_id', how='left', indicator=True)
print(f'after:  {len(check)} rows, ${check["revenue"].sum():.2f}')
print()
print(check['_merge'].value_counts())

before: 8 rows, $183.50
after:  10 rows, $261.50

_merge
both          9
left_only     1
right_only    0
Name: count, dtype: int64


Eight orders went in and nine came out, and the revenue total moved. Both symptoms point at the same cause, and it is in the `vendors` table, not in the orders.

Find it before you go further — everything downstream inherits this bug.

In [3]:
# Which vendor_id appears more than once in the vendor list?
print(vendors['vendor_id'].value_counts())
print()
print('duplicated vendor rows:', vendors.duplicated().sum())

vendor_id
V-18    2
V-01    1
V-05    1
V-10    1
Name: count, dtype: int64

duplicated vendor rows: 1


### Build 1 — fix the vendor list, then merge

**TODO:** drop the duplicate vendor row, then join it onto `orders` with an indicator. Your merge must come out at **8 rows** with the revenue total unchanged from the baseline. Print both to prove it.

In [4]:
orders["revenue"] = orders["qty"] * orders["price"]
clean_vendors = vendors.drop_duplicates(subset="vendor_id", keep="first")
joined = orders.merge(clean_vendors, on="vendor_id", how="left", indicator=True)

print("rows after merge:", len(joined))
print("baseline rows:", baseline_rows)
print("revenue after merge:", joined["revenue"].sum())
print("baseline revenue:", baseline_revenue)
print("same rows as baseline?", len(joined) == baseline_rows)
print("same revenue as baseline?", round(joined["revenue"].sum(), 2) == round(baseline_revenue, 2))
print(joined["_merge"].value_counts())

rows after merge: 8
baseline rows: 8
revenue after merge: 183.5
baseline revenue: 183.5
same rows as baseline? True
same revenue as baseline? True
_merge
both          7
left_only     1
right_only    0
Name: count, dtype: int64


### Build 2 — handle the vendor nobody has heard of

One order belongs to a vendor that is not on the roster. You have three options, and this is a judgment call:

1. Drop it — clean report, understated revenue.
2. Keep it with a blank name — honest, but it will show up as `NaN` in every chart.
3. Label it `'Unknown vendor'` and keep it in a zone called `'Unassigned'`.

**TODO:** pick one, implement it, and write one sentence saying why. Print how much revenue the decision affects either way.

In [5]:
orders["revenue"] = orders["qty"] * orders["price"]
joined = orders.merge(clean_vendors, on="vendor_id", how="left")

unknown_rows = joined[joined["vendor_name"].isna()]
unknown_revenue = unknown_rows["revenue"].sum()

print("unknown vendor rows:")
print(unknown_rows[["order_id", "vendor_id", "revenue"]])
print("unknown revenue:", unknown_revenue)

joined.loc[joined["vendor_name"].isna(), "vendor_name"] = "Unknown vendor"
joined.loc[joined["vendor_name"].isna(), "zone"] = "Unassigned"

unknown vendor rows:
   order_id vendor_id  revenue
7         8      V-42     15.0
unknown revenue: 15.0


**My decision, and why:** I labeled the missing vendor as "Unknown vendor" in an "Unassigned" zone, because keeping it visible is more honest than dropping it, and it avoids creating a blank or missing value problem in downstream reporting.

### Build 3 — the per-vendor summary

**TODO:** one row per vendor, with:

- `orders` — how many orders
- `units` — total quantity
- `revenue` — total revenue
- `avg_ticket` — average revenue per order, rounded to 2 decimals

Sorted by revenue, highest first. Use `.agg()` with named outputs so the columns come out with the names above.

In [6]:
vendor_summary = joined.groupby(["vendor_id", "vendor_name", "zone"]).agg(
    orders=("order_id", "count"),
    units=("qty", "sum"),
    revenue=("revenue", "sum")
).reset_index()

vendor_summary["avg_ticket"] = vendor_summary["revenue"] / vendor_summary["orders"]
vendor_summary["avg_ticket"] = vendor_summary["avg_ticket"].round(2)
vendor_summary = vendor_summary.sort_values("revenue", ascending=False)

print(vendor_summary[["vendor_name", "orders", "units", "revenue", "avg_ticket"]])

       vendor_name  orders  units  revenue  avg_ticket
3  Rally Rain Gear       2     13     78.0       39.00
2  Cav Merch North       2      2     36.0       18.00
0     Hoos Burgers       2      5     28.5       14.25
1    Rotunda Tacos       1      4     26.0       26.00


### Build 4 — did each zone hit its target?

**TODO:** total revenue by zone, join `targets` on, and add a `hit_target` boolean column. Then print a one-line sentence for each zone that a manager could read.

In [7]:
zone_revenue = joined.groupby("zone")["revenue"].sum().reset_index()
zone_revenue = zone_revenue.rename(columns={"revenue": "zone_revenue"})

zone_report = zone_revenue.merge(targets, on="zone", how="left")
zone_report["hit_target"] = zone_report["zone_revenue"] >= zone_report["revenue_target"]

print(zone_report[["zone", "zone_revenue", "revenue_target", "hit_target"]])

for index, row in zone_report.iterrows():
    status = "hit" if row["hit_target"] else "missed"
    print(row["zone"], "revenue:", row["zone_revenue"], "target:", row["revenue_target"], "status:", status)

  zone  zone_revenue  revenue_target  hit_target
0    A          64.5              80       False
1    B          26.0              25        True
2    C          78.0              60        True
A revenue: 64.5 target: 80 status: missed
B revenue: 26.0 target: 25 status: hit
C revenue: 78.0 target: 60 status: hit


### Build 5 — the one number that matters

**TODO:** rain gear is the thing we can actually act on. Print total poncho units sold and what share of overall revenue they represent, as a percentage rounded to one decimal.

In [8]:
poncho = joined[joined["item"].str.lower() == "rain poncho"]
poncho_units = poncho["qty"].sum()
poncho_revenue = poncho["revenue"].sum()
overall_revenue = joined["revenue"].sum()
share = (poncho_revenue / overall_revenue) * 100

print("Rain poncho units sold:", poncho_units)
print("Rain poncho revenue:", poncho_revenue)

print("Overall revenue:", overall_revenue) 
print("Rain poncho share of revenue:", round(share, 1), "%")

Rain poncho units sold: 13
Rain poncho revenue: 78.0
Overall revenue: 183.5
Rain poncho share of revenue: 42.5 %


---

## Checkpoint (participation)

Report your row count and revenue total after the merge, and what you decided to do with the unknown vendor.

Work the last few minutes in groups of 4–5, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Wednesday 11:59pm ET**. One submission per person, not per group.

In [9]:
# Checkpoint
rows_after_merge = 8
revenue_after_merge = baseline_revenue
unknown_vendor_call = "Label the missing vendor as 'Unknown vendor' in an 'Unassigned' zone so it stays visible and honest without distorting the clean roster."

print('rows after merge:', rows_after_merge)
print('revenue after merge:', revenue_after_merge)
print('unknown vendor:', unknown_vendor_call)

rows after merge: 8
revenue after merge: 183.5
unknown vendor: Label the missing vendor as 'Unknown vendor' in an 'Unassigned' zone so it stays visible and honest without distorting the clean roster.
